<a href="https://colab.research.google.com/github/vishal9198/genAi-Labs/blob/main/step_back.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ==========================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==========================================
!pip install -qU langchain langchain-openai langchain-community langchain-chroma tiktoken beautifulsoup4

import os
from google.colab import userdata

# Automatically loads the secret saved in the left sidebar
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# ==========================================
# 2. INDEXING (Creating a Retriever for Testing)
# ==========================================
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

# Load a sample technical blog post to serve as our knowledge base
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header")))
)
docs = loader.load()

# Split the document into searchable chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

# Embed the chunks and store them in a local Chroma vector database
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# ==========================================
# 3. STEP-BACK QUERY GENERATION
# ==========================================
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define few-shot examples to teach the LLM how to abstract a question.
# We provide an 'input' (specific) and an 'output' (broad/step-back).
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]

# Create a formatting template for the few-shot examples
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# Compile the examples into a formal Few-Shot prompt object
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Build the main prompt: System instructions -> Few-Shot Examples -> Actual User Question
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:",
    ),
    few_shot_prompt,
    ("user", "{question}"),
])

# Create the LCEL chain to generate the step-back query
# It passes the prompt to the LLM (temperature=0 for deterministic output) and parses the string.
generate_queries_step_back = prompt | ChatOpenAI(temperature=0) | StrOutputParser()

# Test the query generation standalone
question = "What is task decomposition for LLM agents?"
print("Original Question:", question)
print("Step-Back Question:", generate_queries_step_back.invoke({"question": question}))
print("-" * 50)

# ==========================================
# 4. DUAL-RETRIEVAL & FINAL SYNTHESIS
# ==========================================
from langchain_core.runnables import RunnableLambda

# Define the final response template. It explicitly takes both the normal context
# (specific details) and the step-back context (broad concepts).
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question.
Your response should be comprehensive and not contradicted with the following context if they are relevant.
Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

# Build the final LCEL chain using a dictionary to route multiple parallel processes
chain = (
    {
        # 1. Normal Context: Take the raw "question" string and pass it directly to the retriever
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,

        # 2. Step-Back Context: Pass the raw question into our step-back generator chain,
        #    then pipe that new broad question into the retriever
        "step_back_context": generate_queries_step_back | retriever,

        # 3. Question: Simply pass the raw question through to fill the {question} variable in the prompt
        "question": lambda x: x["question"],
    }
    # Once the dictionary is populated (retrievals are done), pass all three variables into the response prompt
    | response_prompt
    # Send the populated prompt to the LLM to generate the final answer
    | ChatOpenAI(temperature=0)
    # Extract the raw text from the LLM output message
    | StrOutputParser()
)

# Execute the full pipeline
final_answer = chain.invoke({"question": question})
print("Final Answer:\n", final_answer)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************-96w. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}